In [1]:
import pandas as pd
import numpy as np
from IPython.display import clear_output
import arviz as az
import matplotlib
import matplotlib.pyplot as plt

az.style.use(["science", "arviz-doc", "tableau-colorblind10"])
nice_fonts = {
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
    "axes.labelsize": 8,
    "font.size": 8,
    "legend.fontsize": 8,
    "legend.frameon": False,
}
matplotlib.rcParams.update(nice_fonts)
import matplotlib_inline.backend_inline

matplotlib_inline.backend_inline.set_matplotlib_formats(
    "svg", "pdf", "retina"
)  # For export

## Load dataset
We first load and preprocess Albania load demand data available in this  [here](https://data.dtu.dk/articles/dataset/Albanian_national_electricity_consumption_and_weather_conditions_for_2016-2019/22786922). 


In [2]:
data = pd.read_parquet("../data/albania_res.parquet")
data.index = pd.to_datetime(data.index, utc="UTC")

## Create data pipeline

Then we  sett up data transformation and scaling for time series forecasting. The goal is to prepare the dataset for modeling by defining the necessary parameters for the transformation and scaling processes.

For this purpose we use ``DatasetObjective`` A utility from mlpforecast for managing dataset objectives, crucial for time series forecasting.

In [3]:
from mlpforecast.data.transform import DatasetObjective

To achive this we define set of variables as follow

- target_series: The target variable to be predicted; in this case, "NetLoad".
- unknown_features: Any fnumerical eatures not observed in the future etc demand price.
- calendar_variables: Features derived from the calendar, such as hour and session.
- known_calendar_features: Precomputed features that represent the calendar variables.
- known_continuous_features: Continuous features used in the modeling process, including lagged variables and temperature.
- input_window_size: The size of the input window for the model (96 time steps).
- forecast_horizon: The forecasting horizon (48 time steps).

In [4]:
common_params = {
    "target_series": ["NetLoad"],
    "unknown_features": [],
    "calendar_variables": ["HOUR", "Session"],
    "known_calendar_features": ["HOUR-cosin", "Session-cosin"],
    "known_continuous_features": ["NetLoad_lag_48", "NetLoad_lag_336", "Temperature"],
    "input_window_size": 96,
    "forecast_horizon": 48,
}

Then we specify how we want to transform the data adding more features such as lagging or rolling features and scaling function using sklaern processing scaler

In [5]:
from sklearn.preprocessing import (
    MinMaxScaler,
    RobustScaler,
    PowerTransformer,
    StandardScaler,
)

data_params = {
    "input_scaler": PowerTransformer('yeo-johnson'),
    "target_scaler": PowerTransformer('box-cox'),
    "lags": [1, 7],
    "windows": [],
    "window_funcs": ["mean"],
    "period": "30min",
    "date_column": "timestamp",
}

We update the parameters and define the  ``DatasetObjective``. This line ```data_params.update(common_params)`` merges common_params into data_params, ensuring all relevant settings are included for the dataset transformation.

In [6]:
data_params.update(common_params)
ds = DatasetObjective(**data_params)
clear_output()


Since the dataset objective is an instance of sklearn object it implement two main function ```fit``` and ```transform```. The ```fit```  fit the DatasetObjective object (ds) to the provided data, ensuring that the transformations and scaling parameters are applied correctly to the dataset.

In [7]:
ds.fit(data.reset_index())
clear_output()

This fitted data processing pipeline associated with the DatasetObjective instance contains all the transformation steps that will be applied to the dataset.

In [ ]:
ds.data_pipeline

To apply the fitted transform to the data we call ```transform``` function. The transformed data will produce sequences of features and targets that can be directly consumed by a model, streamlining the forecasting process.

In [9]:
x, y = ds.transform(data.reset_index())

After fitting and transforming the dataset, you can proceed to analyze the transformed data, and train forecasting models using the prepared feature and target sequences.

In [11]:
from mlpforecast.forecaster.regressor import RegressorForecast

In [ ]:
m=RegressorForecast(data_pipeline=ds, model_type='LinearReg')
m.fit(train_df=data.reset_index()[:48*7*4])

In [ ]:
#pred=m.predict(data.reset_index()[48*7*4 : 48*7*4 + 48 * 7 * 2], daily_feature=True)
pred_df, metrics_df=m.evaluate(data.reset_index()[48*7*4 : 48*7*4 + 48 * 7 * 2])

In [ ]:
from mlpforecast.plot.visual_functions import plot_prediction

fig, ax = plt.subplots(1, 1, figsize=(9, 2))
ax = plot_prediction(ax, true=pred_df["NetLoad"].values, mu=pred_df["NetLoad_forecast"].values)
ax.set_ylim(0, 1500)
ax.set_ylabel("Power (MWh)")
fig.tight_layout(pad=1.08, h_pad=0.5, w_pad=0.5)

In [ ]:
met = ["RMSE", "MAE", "MAPE(%)", "CORR", "R2-error", "NBIAS", "SMAPE(%)"]
metrics_df.groupby("target")[met].mean().round(2)

In [210]:
m=FEDformer(n_target_series=1, n_known_calendar_features=2, n_known_continuous_features=1,
           n_unknown_features=0)

In [211]:
m(torch.randn(4, 144, 4))

tensor([[[-0.0212],
         [-0.3317],
         [ 0.1200],
         [ 0.2278],
         [-0.1876],
         [ 0.0266],
         [-0.5225],
         [ 0.1963],
         [ 0.1963],
         [-0.1149],
         [-0.3114],
         [-0.2303],
         [-0.2729],
         [ 0.3111],
         [ 0.2974],
         [-0.1899],
         [-0.7861],
         [ 0.1671],
         [ 0.3833],
         [ 0.0518],
         [ 0.1844],
         [ 0.4313],
         [ 0.4414],
         [-0.1182],
         [ 0.2047],
         [ 0.0874],
         [ 0.0369],
         [ 0.6779],
         [ 0.1621],
         [ 0.1980],
         [-0.0724],
         [ 0.5047],
         [-0.2000],
         [ 0.1758],
         [ 0.1941],
         [-0.0099],
         [-0.0971],
         [ 0.3379],
         [-0.2137],
         [-0.1842],
         [-0.2137],
         [ 0.3823],
         [ 0.1938],
         [-0.2003],
         [-0.0969],
         [ 0.0543],
         [ 0.2060],
         [ 0.6041]],

        [[ 0.1558],
         [-0.1789]